# Library management system

Rules used in this notebook:
- a member can borrow max 3 books at a time
- a book can be kept for 14 days
- after that the fine is 5 rupees per day

## imports

In [ ]:
import json
import os
from datetime import date, timedelta

## Book class

In [ ]:
class Book:
    def __init__(self, book_id, title, author, copies=1):
        self.book_id = book_id
        self.title = title
        self.author = author
        self.copies = copies
        self.available = copies

    def __str__(self):
        return f"{self.book_id} | {self.title} | {self.author} | {self.available}/{self.copies} available"

In [ ]:
b = Book(1, "Wings of Fire", "A P J Abdul Kalam", 2)
print(b)

## Member class

In [ ]:
class Member:
    def __init__(self, member_id, name):
        self.member_id = member_id
        self.name = name
        self.borrowed = {}   # book_id -> date it was issued (as text)
        self.fine_paid = 0

    def __str__(self):
        return f"{self.member_id} | {self.name} | books with member: {len(self.borrowed)}"

In [ ]:
m = Member(101, "Rahul")
print(m)

## Library class

First the basic part, adding and removing books and members.

In [ ]:
class Library:
    MAX_BOOKS = 3
    LOAN_DAYS = 14
    FINE_PER_DAY = 5

    def __init__(self):
        self.books = {}
        self.members = {}
        self.history = []    # every issue / return goes here

    def add_book(self, book_id, title, author, copies=1):
        if book_id in self.books:
            # same book again, just add copies
            self.books[book_id].copies += copies
            self.books[book_id].available += copies
            print("added", copies, "copy/copies of", self.books[book_id].title)
        else:
            self.books[book_id] = Book(book_id, title, author, copies)
            print("book added:", title)

    def remove_book(self, book_id):
        if book_id not in self.books:
            print("no such book")
            return
        book = self.books[book_id]
        if book.available != book.copies:
            print("cannot remove, some copies are issued")
            return
        del self.books[book_id]
        print("removed:", book.title)

    def register_member(self, member_id, name):
        if member_id in self.members:
            print("member id already used")
            return
        self.members[member_id] = Member(member_id, name)
        print("member registered:", name)

Now searching and showing things. The rest of the methods are attached to the class in separate cells so each cell stays short.

In [ ]:
def search_books(self, keyword):
    keyword = keyword.lower()
    found = []
    for book in self.books.values():
        if keyword in book.title.lower() or keyword in book.author.lower():
            found.append(book)
    return found

def show_books(self):
    if not self.books:
        print("library is empty")
        return
    for book in self.books.values():
        print(book)

def show_members(self):
    if not self.members:
        print("no members yet")
        return
    for member in self.members.values():
        print(member)

Library.search_books = search_books
Library.show_books = show_books
Library.show_members = show_members

Issue and return. Dates can be passed in manually so we can test the fine without waiting for 2 weeks.

In [ ]:
def issue_book(self, member_id, book_id, issue_date=None):
    if member_id not in self.members:
        print("member not found")
        return
    if book_id not in self.books:
        print("book not found")
        return

    member = self.members[member_id]
    book = self.books[book_id]

    if book_id in member.borrowed:
        print(member.name, "already has this book")
        return
    if len(member.borrowed) >= self.MAX_BOOKS:
        print(member.name, "has reached the limit of", self.MAX_BOOKS, "books")
        return
    if book.available == 0:
        print("no copy available right now")
        return

    issue_date = issue_date or date.today()
    book.available -= 1
    member.borrowed[book_id] = issue_date.isoformat()
    self.history.append({
        "member_id": member_id, "book_id": book_id,
        "title": book.title, "issued": issue_date.isoformat(),
        "returned": None, "fine": 0,
    })
    due = issue_date + timedelta(days=self.LOAN_DAYS)
    print(f"{book.title} issued to {member.name}, due on {due}")

def return_book(self, member_id, book_id, return_date=None):
    if member_id not in self.members or book_id not in self.books:
        print("wrong member or book id")
        return

    member = self.members[member_id]
    if book_id not in member.borrowed:
        print(member.name, "does not have this book")
        return

    return_date = return_date or date.today()
    issued = date.fromisoformat(member.borrowed[book_id])
    days_kept = (return_date - issued).days
    late_days = max(0, days_kept - self.LOAN_DAYS)
    fine = late_days * self.FINE_PER_DAY

    del member.borrowed[book_id]
    self.books[book_id].available += 1
    member.fine_paid += fine

    # update the matching open record in history
    for record in self.history:
        if (record["member_id"] == member_id and record["book_id"] == book_id
                and record["returned"] is None):
            record["returned"] = return_date.isoformat()
            record["fine"] = fine
            break

    print(f"{self.books[book_id].title} returned by {member.name}")
    if fine > 0:
        print(f"late by {late_days} days, fine = Rs {fine}")

Library.issue_book = issue_book
Library.return_book = return_book

Saving and loading from a json file, so the data is not lost when the notebook is closed.

In [ ]:
def save(self, filename="library_data.json"):
    data = {
        "books": [vars(b) for b in self.books.values()],
        "members": [vars(m) for m in self.members.values()],
        "history": self.history,
    }
    with open(filename, "w") as f:
        json.dump(data, f, indent=2)
    print("saved to", filename)

def load(self, filename="library_data.json"):
    if not os.path.exists(filename):
        print("file not found")
        return
    with open(filename) as f:
        data = json.load(f)

    self.books = {}
    for b in data["books"]:
        book = Book(b["book_id"], b["title"], b["author"], b["copies"])
        book.available = b["available"]
        self.books[book.book_id] = book

    self.members = {}
    for m in data["members"]:
        member = Member(m["member_id"], m["name"])
        member.borrowed = m["borrowed"]
        member.fine_paid = m["fine_paid"]
        self.members[member.member_id] = member

    self.history = data["history"]
    print("loaded", len(self.books), "books and", len(self.members), "members")

Library.save = save
Library.load = load

## using the system

In [ ]:
lib = Library()

add some books

In [ ]:
lib.add_book(1, "Wings of Fire", "A P J Abdul Kalam", 2)
lib.add_book(2, "The Alchemist", "Paulo Coelho", 1)
lib.add_book(3, "Python Crash Course", "Eric Matthes", 3)
lib.add_book(4, "Ikigai", "Hector Garcia", 1)
lib.add_book(5, "Atomic Habits", "James Clear", 2)

In [ ]:
# adding the same id again only adds copies
lib.add_book(2, "The Alchemist", "Paulo Coelho", 1)

In [ ]:
lib.show_books()

register members

In [ ]:
lib.register_member(101, "Rahul")
lib.register_member(102, "Priya")
lib.register_member(103, "Aman")
lib.register_member(101, "Someone else")

In [ ]:
lib.show_members()

search

In [ ]:
for book in lib.search_books("python"):
    print(book)

In [ ]:
for book in lib.search_books("paulo"):
    print(book)

issue books

In [ ]:
lib.issue_book(101, 1)
lib.issue_book(101, 3)
lib.issue_book(102, 1)

In [ ]:
# only 2 copies of book 1, so the third person should get an error
lib.issue_book(103, 1)

In [ ]:
# same book twice to same member
lib.issue_book(101, 1)

In [ ]:
# rahul has 2 books now, the 3rd works but the 4th should fail
lib.issue_book(101, 2)
lib.issue_book(101, 4)

In [ ]:
lib.show_books()

return books

In [ ]:
# returned on time, no fine
lib.return_book(102, 1)

In [ ]:
# book 3 was issued today but we pretend it comes back after 20 days
late_day = date.today() + timedelta(days=20)
lib.return_book(101, 3, return_date=late_day)

In [ ]:
# returning something the member doesnt have
lib.return_book(103, 5)

remove a book

In [ ]:
# book 1 still has one copy with rahul
lib.remove_book(1)

In [ ]:
# nobody has book 5, so it can go
lib.remove_book(5)
lib.show_books()

## reports

In [ ]:
# who has what right now
for member in lib.members.values():
    print(member.name)
    if not member.borrowed:
        print("   nothing")
    for book_id, issued_on in member.borrowed.items():
        print("  ", lib.books[book_id].title, "issued on", issued_on)

In [ ]:
# total fine collected
total = sum(m.fine_paid for m in lib.members.values())
print("total fine collected: Rs", total)

history can go into a pandas dataframe for easy filtering

In [ ]:
import pandas as pd

history_df = pd.DataFrame(lib.history)
history_df

In [ ]:
# books that are still not returned
history_df[history_df["returned"].isnull()]

In [ ]:
import matplotlib.pyplot as plt

history_df["title"].value_counts().plot(kind="bar")
plt.title("Times each book was issued")
plt.ylabel("count")
plt.xticks(rotation=30)
plt.show()

## save and load

In [ ]:
lib.save()

In [ ]:
new_lib = Library()
new_lib.load()
new_lib.show_books()

## menu version

This part is for running the system with input(). Run the cell and type the option number. Choose 0 to stop.

In [ ]:
def menu(lib):
    while True:
        print()
        print("1. add book")
        print("2. show books")
        print("3. register member")
        print("4. issue book")
        print("5. return book")
        print("6. search")
        print("7. save")
        print("0. exit")
        choice = input("choice: ").strip()

        if choice == "1":
            lib.add_book(int(input("book id: ")), input("title: "),
                         input("author: "), int(input("copies: ")))
        elif choice == "2":
            lib.show_books()
        elif choice == "3":
            lib.register_member(int(input("member id: ")), input("name: "))
        elif choice == "4":
            lib.issue_book(int(input("member id: ")), int(input("book id: ")))
        elif choice == "5":
            lib.return_book(int(input("member id: ")), int(input("book id: ")))
        elif choice == "6":
            for book in lib.search_books(input("keyword: ")):
                print(book)
        elif choice == "7":
            lib.save()
        elif choice == "0":
            print("bye")
            break
        else:
            print("wrong choice")

In [ ]:
# uncomment to try it
# menu(lib)